In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import TopKCategoricalAccuracy

In [ ]:
df = pd.read_csv('/content/qoute_dataset.csv')

In [ ]:
df.head()

In [ ]:
texts = df['quote'].astype(str).tolist()

In [ ]:
import string

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

texts = [clean_text(t) for t in texts]

In [ ]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

total_words = len(tokenizer.word_index) + 1

In [ ]:
input_sequences = []

for line in texts:

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):

        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

In [ ]:
max_len = max([len(x) for x in input_sequences])

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding='pre'
)

In [ ]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [ ]:
model = Sequential()

model.add(
    Embedding(
        input_dim=total_words,
        output_dim=200
    )
)

model.add(
    Bidirectional(
        LSTM(128, return_sequences=True)
    )
)

model.add(Dropout(0.3))

model.add(LSTM(128))

model.add(Dropout(0.3))

model.add(Dense(total_words, activation='softmax'))

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)
    ]
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128
)

In [ ]:
results = model.evaluate(X_val, y_val)

print("Loss:", results[0])
print("Accuracy:", results[1])
print("Top5 Accuracy:", results[2])

In [ ]:
loss = results[0]
perplexity = np.exp(loss)

print("Perplexity:", perplexity)

In [ ]:
def predict_next_word(seed_text, temperature=1.0):

    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_len-1,
        padding='pre'
    )

    preds = model.predict(token_list)[0]

    preds = np.log(preds + 1e-10) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)

    predicted_index = np.random.choice(len(preds), p=preds)

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

In [ ]:
def generate_text(seed_text, next_words=10):

    for _ in range(next_words):

        next_word = predict_next_word(seed_text)

        seed_text += " " + next_word

    return seed_text

In [ ]:
print(generate_text("life is", 10))